In [ ]:
#tag category classification
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import get_tag_columns, make_sample_dataset, preprocess_dataframe, train_val_test_split
from tagging.naive_bayes import NBTaggerModel
from tagging.svm_model import SVMTaggerModel
from tagging.random_forest import RFTaggerModel
from tagging.cnn_model import CNNTaggerModel
from tagging.mlp_model import MLPTaggerModel
from evaluation.metrics import (
    tag_report, plot_per_label_f1,
    plot_tag_model_comparison, build_comparison_table,
)

sns.set_theme(style='whitegrid')
%matplotlib inline
os.makedirs('../data/figures', exist_ok=True)

In [ ]:
#load splits
try:
    train = pd.read_parquet('../data/train.parquet')
    test  = pd.read_parquet('../data/test.parquet')
    print(f'Loaded splits: train={len(train):,} test={len(test):,}')
except FileNotFoundError:
    print('Parquet not found — using synthetic data.')
    df = preprocess_dataframe(make_sample_dataset(n=3000))
    train, _, test = train_val_test_split(df)

tag_cols = get_tag_columns(train)
TAG_NAMES = [c.replace('tag_', '').replace('_', ' ').title() for c in tag_cols]
print('Tags:', TAG_NAMES)

X_train = train['text_clean'].tolist()
Y_train = train[tag_cols].values
X_test  = test['text_clean'].tolist()
Y_test  = test[tag_cols].values

In [ ]:
#NB OneVsRest
nb = NBTaggerModel()
nb.fit(X_train, Y_train, TAG_NAMES)
nb_preds, nb_infer = nb.timed_predict(X_test)
nb_results = tag_report(Y_test, nb_preds, TAG_NAMES, model_name='Naive Bayes')
nb_results['train_time'] = nb.train_time_
nb_results['infer_time'] = nb_infer

In [ ]:
#SVM LinearSVC + OvR
svm = SVMTaggerModel()
svm.fit(X_train, Y_train, TAG_NAMES)
svm_preds, svm_infer = svm.timed_predict(X_test)
svm_results = tag_report(Y_test, svm_preds, TAG_NAMES, model_name='SVM')
svm_results['train_time'] = svm.train_time_
svm_results['infer_time'] = svm_infer

In [ ]:
#random Forest OvR
rf = RFTaggerModel(n_estimators=200)
rf.fit(X_train, Y_train, TAG_NAMES)
rf_preds, rf_infer = rf.timed_predict(X_test)
rf_results = tag_report(Y_test, rf_preds, TAG_NAMES, model_name='Random Forest')
rf_results['train_time'] = rf.train_time_
rf_results['infer_time'] = rf_infer

In [ ]:
#20k for CPU training (CNN and MLP only)
import random; random.seed(42)
idx = random.sample(range(len(X_train)), 20_000)
X_train_s = [X_train[i] for i in idx]
Y_train_s = Y_train[idx]
print(f'Train sample: {len(X_train_s):,}  Test (full): {len(X_test):,}')

In [ ]:
#CNN for Text word embedding
cnn = CNNTaggerModel(embed_dim=128, num_filters=128, filter_sizes=(2, 3, 4), epochs=3)
cnn.fit(X_train_s, Y_train_s, TAG_NAMES)
cnn_preds, cnn_infer = cnn.timed_predict(X_test)
cnn_results = tag_report(Y_test, cnn_preds, TAG_NAMES, model_name='CNN (Text)')
cnn_results['train_time'] = cnn.train_time_
cnn_results['infer_time'] = cnn_infer

# CNN filter visualization 
sample_text = X_test[0]
activations = cnn.get_filter_activations(sample_text)
fig, axes = plt.subplots(1, len(activations), figsize=(14, 3))
for ax, (name, vals) in zip(axes, activations.items()):
    ax.bar(range(len(vals)), vals, color='steelblue')
    ax.set_title(f'{name}\nmax activation per filter')
    ax.set_xlabel('Filter index')
plt.suptitle('CNN Filter Activations', fontsize=12)
plt.tight_layout()
plt.savefig('../data/figures/cnn_filter_activations.png', dpi=150)
plt.show()

In [ ]:
#word embedding GloVe
mlp = MLPTaggerModel(
    embed_source='glove-wiki-gigaword-100',
    embed_dim=100,
    hidden_dims=(256, 128),
    epochs=10,
)
mlp.fit(X_train_s, Y_train_s, TAG_NAMES)
mlp_preds, mlp_infer = mlp.timed_predict(X_test)
mlp_results = tag_report(Y_test, mlp_preds, TAG_NAMES, model_name='MLP (GloVe)')
mlp_results['train_time'] = mlp.train_time_
mlp_results['infer_time'] = mlp_infer

In [ ]:
#comparison
all_tag_results = [nb_results, svm_results, rf_results, cnn_results, mlp_results]

comparison_df = build_comparison_table(all_tag_results)
print(comparison_df[['hamming_loss', 'lrap', 'subset_accuracy', 'train_time']].round(4))

plot_per_label_f1(all_tag_results, TAG_NAMES, save_path='../data/figures/per_label_f1.png');
plot_tag_model_comparison(all_tag_results, save_path='../data/figures/tag_comparison.png');

In [ ]:
# TF-IDF vs Word Embedding feature comparison
tfidf_models = ['Naive Bayes', 'SVM', 'Random Forest']
embed_models = ['CNN (Text)', 'MLP (GloVe)']

tfidf_lraps = [r['lrap'] for r in all_tag_results if r['model'] in tfidf_models]
embed_lraps = [r['lrap'] for r in all_tag_results if r['model'] in embed_models]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['TF-IDF avg', 'Embedding avg'],
       [np.mean(tfidf_lraps), np.mean(embed_lraps)],
       color=['#3498db', '#e67e22'])
ax.set_ylabel('LRAP')
ax.set_title('TF-IDF vs Word Embedding Features\n(avg LRAP across models)')
plt.tight_layout()
plt.savefig('../data/figures/tfidf_vs_embedding.png', dpi=150)
plt.show()

In [ ]:
# Qualitative — correct vs incorrect predictions
for i in range(min(5, len(X_test))):
    true_tags = [TAG_NAMES[j] for j, v in enumerate(Y_test[i]) if v == 1]
    pred_tags = [TAG_NAMES[j] for j, v in enumerate(svm_preds[i]) if v == 1]
    print(f'Review: {X_test[i][:80]}...')
    print(f'  True: {true_tags}')
    print(f'  SVM:  {pred_tags}')
    print()